In [14]:
import os
import shutil
import pandas as pd
import requests
from PIL import Image
from urllib.request import urlretrieve
from astroquery.sdss import SDSS



In [15]:
# Define the save path within the Colab drive
base_dir = '/content/data'
categories = ['star', 'galaxy', 'quasar', 'nebula', 'planet']

for category in categories:
    os.makedirs(os.path.join(base_dir, category), exist_ok=True)

print(f"Folder structure successfully created at: {base_dir}")

Estructura de carpetas creada con éxito en: /content/data


In [16]:
import os
import requests

base_dir = '/content/data'
categories = ['star', 'galaxy', 'quasar', 'nebula', 'planet']

# 1. Create directories
for cat in categories:
    os.makedirs(os.path.join(base_dir, cat), exist_ok=True)

# 2. Function to download images from the official NASA API
def download_from_nasa_api(query, class_name, limit=50):
    print(f"Downloading '{class_name}' from NASA API...")
    url = f"https://images-api.nasa.gov/search?q={query}&media_type=image"

    try:
        res = requests.get(url, timeout=10).json()
        items = res.get('collection', {}).get('items', [])
        target_dir = os.path.join(base_dir, class_name)

        count = 0
        for item in items:
            if count >= limit:
                break
            links = item.get('links', [])
            if links:
                img_url = links[0].get('href')
                if img_url and img_url.endswith(('.jpg', '.png', '.jpeg')):
                    img_data = requests.get(img_url, timeout=5).content
                    save_path = os.path.join(target_dir, f"nasa_{class_name}_{count}.jpg")
                    with open(save_path, 'wb') as f:
                        f.write(img_data)
                    count += 1
        print(f"Saved {count} images of {class_name} (NASA API)")
    except Exception as e:
        print(f"Error in NASA API for {class_name}: {e}")

# 3. Function to download SDSS cutouts using direct HTTP
def download_sdss_direct(class_name, limit=50):
    print(f"Downloading '{class_name}' from SDSS Cutouts...")
    target_dir = os.path.join(base_dir, class_name)

    # Real astronomical coordinates of dense fields
    coords_map = {
        'star': (180.0, 0.0),
        'galaxy': (200.0, 15.0),
        'quasar': (150.0, 2.0)
    }

    base_ra, base_dec = coords_map.get(class_name, (180.0, 0.0))
    count = 0

    for i in range(limit * 2):
        if count >= limit:
            break
        ra = base_ra + (i * 0.02)
        dec = base_dec + (i * 0.02)
        url = f"https://skyserver.sdss.org/dr16/SkyServerWS/ImgViewing/getjpeg?ra={ra}&dec={dec}&scale=0.4&width=128&height=128"

        try:
            r = requests.get(url, timeout=3)
            if r.status_code == 200 and len(r.content) > 2000:  # Validate it is not an empty image
                with open(os.path.join(target_dir, f"sdss_{class_name}_{count}.jpg"), 'wb') as f:
                    f.write(r.content)
                count += 1
        except Exception:
            continue

    print(f"Saved {count} images of {class_name} (SDSS)")

# DOWNLOAD EXECUTION
# SDSS
download_sdss_direct('star', limit=50)
download_sdss_direct('galaxy', limit=50)
download_sdss_direct('quasar', limit=50)

# NASA API
download_from_nasa_api('nebula', 'nebula', limit=50)
download_from_nasa_api('planet', 'planet', limit=50)
download_from_nasa_api('galaxy', 'galaxy', limit=30)  # Mix with NASA galaxies

Descargando 'star' desde SDSS Cutouts...
  ✅ Guardadas 0 imágenes de star (SDSS)
Descargando 'galaxy' desde SDSS Cutouts...
  ✅ Guardadas 0 imágenes de galaxy (SDSS)
Descargando 'quasar' desde SDSS Cutouts...
  ✅ Guardadas 0 imágenes de quasar (SDSS)
Descargando 'nebula' desde la API de la NASA...
  ✅ Guardadas 50 imágenes de nebula (NASA API)
Descargando 'planet' desde la API de la NASA...
  ✅ Guardadas 50 imágenes de planet (NASA API)
Descargando 'galaxy' desde la API de la NASA...
  ✅ Guardadas 30 imágenes de galaxy (NASA API)


In [17]:
nebula_urls = [
    "https://apod.nasa.gov/apod/image/2312/OrionCluster_Hubble_960.jpg",
    "https://apod.nasa.gov/apod/image/2011/CrabNebula_Hubble_960.jpg",
    "https://apod.nasa.gov/apod/image/2208/RingNebula_Webb_960.jpg",
    "https://images-assets.nasa.gov/image/PIA04215/PIA04215~orig.jpg"
]

planet_urls = [
    "https://apod.nasa.gov/apod/image/2109/Jupiter_Hubble_960.jpg",
    "https://apod.nasa.gov/apod/image/2303/Saturn_Cassini_960.jpg",
    "https://images-assets.nasa.gov/image/PIA01492/PIA01492~orig.jpg",
    "https://images-assets.nasa.gov/image/PIA00013/PIA00013~orig.jpg"
]

def download_urls(url_list, class_name):
    target_dir = os.path.join(base_dir, class_name)
    for i, url in enumerate(url_list):
        save_path = os.path.join(target_dir, f"nasa_{class_name}_{i}.jpg")
        try:
            urlretrieve(url, save_path)
        except Exception as e:
            print(f"Error downloading {url}: {e}")

download_urls(nebula_urls, 'nebula')
download_urls(planet_urls, 'planet')
print("Nebulae and planets downloaded.")

Error descargando https://apod.nasa.gov/apod/image/2312/OrionCluster_Hubble_960.jpg: HTTP Error 404: Not Found
Error descargando https://apod.nasa.gov/apod/image/2011/CrabNebula_Hubble_960.jpg: HTTP Error 404: Not Found
Error descargando https://apod.nasa.gov/apod/image/2208/RingNebula_Webb_960.jpg: HTTP Error 404: Not Found
Error descargando https://apod.nasa.gov/apod/image/2109/Jupiter_Hubble_960.jpg: HTTP Error 404: Not Found
Error descargando https://apod.nasa.gov/apod/image/2303/Saturn_Cassini_960.jpg: HTTP Error 404: Not Found
Nebulosas y planetas descargados.


In [18]:
print("IMAGE SUMMARY IN GOOGLE COLAB")
for category in categories:
    cat_dir = os.path.join(base_dir, category)
    num_files = len([f for f in os.listdir(cat_dir) if f.endswith(('.jpg', '.png'))])
    print(f"Category '{category:<8}': {num_files} images saved")

=== RESUMEN DE IMÁGENES EN GOOGLE COLAB ===
Categoría 'star    ': 0 imágenes guardadas
Categoría 'galaxy  ': 30 imágenes guardadas
Categoría 'quasar  ': 0 imágenes guardadas
Categoría 'nebula  ': 50 imágenes guardadas
Categoría 'planet  ': 50 imágenes guardadas


In [19]:
import os
import requests

base_dir = '/content/data'

def download_missing_from_nasa(query, class_name, limit=50):
    print(f"Downloading additional images for '{class_name}'...")
    url = f"https://images-api.nasa.gov/search?q={query}&media_type=image"
    target_dir = os.path.join(base_dir, class_name)
    os.makedirs(target_dir, exist_ok=True)

    try:
        res = requests.get(url, timeout=10).json()
        items = res.get('collection', {}).get('items', [])
        count = len(os.listdir(target_dir))
        added = 0

        for item in items:
            if added >= limit:
                break
            links = item.get('links', [])
            if links:
                img_url = links[0].get('href')
                if img_url and img_url.endswith(('.jpg', '.png', '.jpeg')):
                    try:
                        img_data = requests.get(img_url, timeout=5).content
                        save_path = os.path.join(target_dir, f"nasa_{class_name}_{count}.jpg")
                        with open(save_path, 'wb') as f:
                            f.write(img_data)
                        count += 1
                        added += 1
                    except Exception:
                        continue
        print(f"Added {added} images to {class_name}")
    except Exception as e:
        print(f"Error processing {class_name}: {e}")

# Download classes that have 0 (and add more to galaxy)
download_missing_from_nasa('star', 'star', limit=50)
download_missing_from_nasa('quasar', 'quasar', limit=50)
download_missing_from_nasa('galaxy', 'galaxy', limit=20)

Descargando imágenes adicionales para 'star'...
  ✅ Añadidas 50 imágenes a star
Descargando imágenes adicionales para 'quasar'...
  ✅ Añadidas 50 imágenes a quasar
Descargando imágenes adicionales para 'galaxy'...
  ✅ Añadidas 20 imágenes a galaxy


In [20]:
import shutil
from google.colab import files

print("Extracting image folders...")
shutil.make_archive('cosmos_dataset', 'zip', '/content/data')

print("Downloading...")
files.download('cosmos_dataset.zip')

Extracting image folders...
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>